# Comparação de Modelos de Machine Learning - DataSUS

Este notebook compara diferentes modelos de machine learning para predição de custos e tempo de permanência em internações hospitalares.

## Objetivos
- Comparar performance de diferentes algoritmos
- Identificar o melhor modelo baseado em R² Score
- Analisar características de cada abordagem

## Problemas Analisados
1. **Predição de Custos** (Regressão)
2. **Predição de Tempo de Permanência** (Regressão)

In [1]:
# Importações necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modelos de Regressão
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Configuração de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. Carregamento e Preparação dos Dados

In [2]:
def load_data_from_db():
    """Carrega dados do banco SQLite usando a mesma query do projeto"""
    db_path = os.path.join('..', 'database', 'internacoes_datasus.db')
    
    if not os.path.exists(db_path):
        print(f"❌ Banco de dados não encontrado em: {db_path}")
        print("Execute: python scripts/create_database.py")
        return None
    
    conn = sqlite3.connect(db_path)
    
    # Query idêntica à usada no dashboard (main.py)
    query = """
        SELECT 
            i.id as internacao_id,
            i.numero_aih,
            i.ano_competencia,
            i.mes_competencia,
            i.data_internacao,
            i.data_saida,
            i.dias_permanencia,
            i.dias_uti_total,
            i.gestacao_risco,
            
            -- Dados do paciente
            p.idade_anos,
            s.descricao as sexo,
            p.codigo_municipio_residencia,
            m.nome as municipio_residencia,

            -- Dados clínicos com descrições
            cid.descricao as diagnostico_principal,
            cid.capitulo as capitulo_cid,
            cid.sensivel_atencao_basica,
            ci.descricao as carater_internacao,
            
            -- Dados do estabelecimento
            e.codigo_cnes,
            esp.descricao as especialidade,
            comp.descricao as complexidade,
            tg.descricao as tipo_gestao,
            
            -- Valores financeiros
            vf.valor_total,
            vf.valor_servicos_hospitalares,
            vf.valor_servicos_profissionais,
            vf.valor_uti,
            vf.valor_em_dolares,
            
            -- Códigos originais para ML
            p.codigo_sexo,
            e.codigo_especialidade,
            e.codigo_complexidade,
            i.codigo_carater_internacao
            
        FROM internacoes i
        LEFT JOIN pacientes p ON i.paciente_id = p.id
        LEFT JOIN municipios m ON p.codigo_municipio_residencia = m.codigo
        LEFT JOIN sexo s ON p.codigo_sexo = s.codigo
        LEFT JOIN cid_diagnosticos cid ON i.codigo_diagnostico_principal = cid.codigo
        LEFT JOIN carater_internacao ci ON printf('%02d', i.codigo_carater_internacao) = ci.codigo
        LEFT JOIN estabelecimentos e ON i.estabelecimento_id = e.id
        LEFT JOIN especialidades esp ON e.codigo_especialidade = esp.codigo
        LEFT JOIN complexidade comp ON e.codigo_complexidade = comp.codigo
        LEFT JOIN tipos_gestao tg ON e.codigo_tipo_gestao = tg.codigo
        LEFT JOIN valores_financeiros vf ON i.id = vf.internacao_id
        
        WHERE i.codigo_diagnostico_principal IS NOT NULL
        AND p.idade_anos IS NOT NULL
        AND vf.valor_total IS NOT NULL
        AND vf.valor_total > 0
    """
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    print(f"✅ Dados carregados: {len(df):,} registros")
    print(f"📊 Período: {df['ano_competencia'].min()}-{df['mes_competencia'].min()} a {df['ano_competencia'].max()}-{df['mes_competencia'].max()}")
    print(f"💰 Valor médio: R$ {df['valor_total'].mean():,.2f}")
    print(f"⏱️ Permanência média: {df['dias_permanencia'].mean():.1f} dias")
    return df

# Carregar dados
df = load_data_from_db()
if df is not None:
    print(f"\n📋 Shape dos dados: {df.shape}")
    print(f"\n📝 Colunas disponíveis: {len(df.columns)}")
    print("\n🔍 Primeiras linhas:")
    display(df.head())

✅ Dados carregados: 118,496 registros
📊 Período: 2025-1 a 2025-3
💰 Valor médio: R$ 1,931.74
⏱️ Permanência média: 3.7 dias

📋 Shape dos dados: (118496, 30)

📝 Colunas disponíveis: 30

🔍 Primeiras linhas:


,internacao_id,numero_aih,ano_competencia,mes_competencia,data_internacao,data_saida,dias_permanencia,dias_uti_total,gestacao_risco,idade_anos,...,tipo_gestao,valor_total,valor_servicos_hospitalares,valor_servicos_profissionais,valor_uti,valor_em_dolares,codigo_sexo,codigo_especialidade,codigo_complexidade,codigo_carater_internacao
0,403,4124115397622,2025,1,20241226,20241227,1,0,1,57,...,Gestão Estadual,40.38,30.47,9.91,0.0,6.91,1,3,2,2
1,432,4124115397600,2025,1,20241231,20241231,0,0,1,19,...,Gestão Estadual,40.38,30.47,9.91,0.0,6.91,1,3,2,2
2,433,4124115397611,2025,1,20241212,20241213,1,0,1,49,...,Gestão Estadual,40.38,30.47,9.91,0.0,6.91,1,3,2,2
3,534,4124115397655,2025,1,20241230,20241230,0,0,1,24,...,Gestão Estadual,40.38,30.47,9.91,0.0,6.91,1,3,2,2
4,536,4124115397677,2025,1,20241231,20241231,0,0,1,64,...,Gestão Estadual,40.38,30.47,9.91,0.0,6.91,1,3,2,2


In [3]:
def prepare_features(df):
    """Prepara features usando a mesma lógica do dashboard"""
    df_ml = df.copy()
    
    # Features derivadas (idênticas ao machine_learning.py)
    df_ml['tem_uti'] = (df_ml['dias_uti_total'] > 0).astype(int)
    df_ml['valor_por_dia'] = df_ml['valor_total'] / (df_ml['dias_permanencia'] + 1)
    df_ml['idade_grupo'] = pd.cut(df_ml['idade_anos'], bins=[0, 18, 40, 60, 100], labels=[0, 1, 2, 3])
    df_ml['urgencia'] = (df_ml['codigo_carater_internacao'] == '2').astype(int)
    df_ml['permanencia_longa'] = (df_ml['dias_permanencia'] > 7).astype(int)
    df_ml['custo_alto'] = (df_ml['valor_total'] > df_ml['valor_total'].quantile(0.75)).astype(int)
    
    # Tratar valores nulos (mesmo tratamento do dashboard)
    df_ml['codigo_especialidade'] = df_ml['codigo_especialidade'].fillna('99')
    df_ml['codigo_complexidade'] = df_ml['codigo_complexidade'].fillna('9')
    df_ml['gestacao_risco'] = df_ml['gestacao_risco'].fillna(0).astype(int)
    df_ml['sensivel_atencao_basica'] = df_ml['sensivel_atencao_basica'].fillna(0).astype(int)
    df_ml['idade_grupo'] = df_ml['idade_grupo'].fillna(1).astype(int)
    
    # Codificar variáveis categóricas
    categorical_cols = ['codigo_sexo', 'codigo_especialidade', 'codigo_complexidade']
    encoders = {}
    
    for col in categorical_cols:
        if col in df_ml.columns:
            df_ml[col] = df_ml[col].fillna('unknown').astype(str)
            le = LabelEncoder()
            df_ml[col + '_encoded'] = le.fit_transform(df_ml[col])
            encoders[col] = le
    
    return df_ml, encoders

# Preparar dados
if df is not None:
    print("🔧 Preparando features...")
    df_prepared, encoders = prepare_features(df)
    
    # Remover outliers extremos (mesma lógica do dashboard)
    print("🧹 Removendo outliers...")
    df_clean = df_prepared[
        (df_prepared['valor_total'] <= 50000) & 
        (df_prepared['dias_permanencia'] <= 30)
    ].copy()
    
    print(f"✅ Dados após preparação: {len(df_prepared):,} registros")
    print(f"🗑️ Outliers removidos: {len(df_prepared) - len(df_clean):,} registros")
    print(f"📊 Dataset final: {len(df_clean):,} registros")
    
    # Verificar distribuições
    print(f"\n📈 Estatísticas finais:")
    print(f"   Valor total: R$ {df_clean['valor_total'].mean():,.2f} ± R$ {df_clean['valor_total'].std():,.2f}")
    print(f"   Permanência: {df_clean['dias_permanencia'].mean():.1f} ± {df_clean['dias_permanencia'].std():.1f} dias")
    print(f"   Idade: {df_clean['idade_anos'].mean():.1f} ± {df_clean['idade_anos'].std():.1f} anos")
    print(f"   Com UTI: {df_clean['tem_uti'].mean()*100:.1f}%")
    print(f"   Urgência: {df_clean['urgencia'].mean()*100:.1f}%")

🔧 Preparando features...
🧹 Removendo outliers...
✅ Dados após preparação: 118,496 registros
🗑️ Outliers removidos: 1,206 registros
📊 Dataset final: 117,290 registros

📈 Estatísticas finais:
   Valor total: R$ 1,708.31 ± R$ 3,439.38
   Permanência: 3.4 ± 4.4 dias
   Idade: 45.9 ± 23.5 anos
   Com UTI: 11.2%
   Urgência: 68.6%


## 2. Configuração dos Modelos

In [4]:
def get_regression_models():
    """Retorna dicionário com modelos de regressão configurados"""
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(alpha=1.0, random_state=42),
        'Lasso Regression': Lasso(alpha=0.1, random_state=42),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
        'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0),
        'LightGBM': LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbosity=-1),
        'SVR': SVR(kernel='rbf', gamma='scale'),
        'K-Neighbors': KNeighborsRegressor(n_neighbors=5)
    }
    return models

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Avalia um modelo e retorna métricas"""
    try:
        # Treinar modelo
        model.fit(X_train, y_train)
        
        # Fazer predições
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        
        # Calcular métricas
        train_r2 = r2_score(y_train, y_pred_train)
        test_r2 = r2_score(y_test, y_pred_test)
        test_mae = mean_absolute_error(y_test, y_pred_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
        
        # Cross-validation R²
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
        cv_r2_mean = cv_scores.mean()
        cv_r2_std = cv_scores.std()
        
        return {
            'Model': model_name,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'CV_R2_Mean': cv_r2_mean,
            'CV_R2_Std': cv_r2_std,
            'Test_MAE': test_mae,
            'Test_RMSE': test_rmse,
            'Overfitting': train_r2 - test_r2,
            'Predictions': y_pred_test
        }
    except Exception as e:
        print(f"Erro ao treinar {model_name}: {str(e)}")
        return None

print("Modelos configurados:")
models = get_regression_models()
for name in models.keys():
    print(f"- {name}")

Modelos configurados:
- Linear Regression
- Ridge Regression
- Lasso Regression
- ElasticNet
- Decision Tree
- Random Forest
- Extra Trees
- Gradient Boosting
- XGBoost
- LightGBM
- SVR
- K-Neighbors


## 3. Predição de Custos de Internação

In [5]:
if df_clean is not None:
    print("=== PREDIÇÃO DE CUSTOS DE INTERNAÇÃO ===")
    print()
    
    # Definir features para predição de custos
    cost_features = [
        'idade_anos', 'dias_permanencia', 'codigo_sexo_encoded', 'urgencia', 'tem_uti',
        'gestacao_risco', 'codigo_especialidade_encoded', 'codigo_complexidade_encoded',
        'sensivel_atencao_basica', 'mes_competencia', 'idade_grupo'
    ]
    
    # Preparar dados
    X_cost = df_clean[cost_features].fillna(0)
    y_cost = df_clean['valor_total']
    
    # Dividir dados
    X_cost_train, X_cost_test, y_cost_train, y_cost_test = train_test_split(
        X_cost, y_cost, test_size=0.2, random_state=42
    )
    
    # Normalizar features (importante para alguns modelos)
    scaler_cost = StandardScaler()
    X_cost_train_scaled = scaler_cost.fit_transform(X_cost_train)
    X_cost_test_scaled = scaler_cost.transform(X_cost_test)
    
    print(f"Features utilizadas: {len(cost_features)}")
    print(f"Tamanho treino: {len(X_cost_train):,}")
    print(f"Tamanho teste: {len(X_cost_test):,}")
    print(f"Valor médio: R$ {y_cost.mean():,.2f}")
    print(f"Desvio padrão: R$ {y_cost.std():,.2f}")
    print()

=== PREDIÇÃO DE CUSTOS DE INTERNAÇÃO ===

Features utilizadas: 11
Tamanho treino: 93,832
Tamanho teste: 23,458
Valor médio: R$ 1,708.31
Desvio padrão: R$ 3,439.38



In [ ]:
if df_clean is not None:
    # Avaliar todos os modelos para predição de custos
    cost_results = []
    
    print("Treinando modelos para predição de custos...")
    print()
    
    for name, model in get_regression_models().items():
        print(f"Treinando {name}...")
        
        # Usar dados normalizados para modelos que precisam
        if name in ['SVR', 'K-Neighbors', 'Linear Regression', 'Ridge Regression', 'Lasso Regression', 'ElasticNet']:
            X_train_use = X_cost_train_scaled
            X_test_use = X_cost_test_scaled
        else:
            X_train_use = X_cost_train
            X_test_use = X_cost_test
        
        result = evaluate_model(model, X_train_use, X_test_use, y_cost_train, y_cost_test, name)
        if result:
            cost_results.append(result)
    
    # Criar DataFrame com resultados
    cost_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Predictions'} for r in cost_results])
    cost_df = cost_df.sort_values('Test_R2', ascending=False)
    
    print("\n=== RESULTADOS - PREDIÇÃO DE CUSTOS ===")
    print()
    
    # Exibir resultados formatados
    display_df = cost_df.copy()
    display_df['Train_R2'] = display_df['Train_R2'].round(4)
    display_df['Test_R2'] = display_df['Test_R2'].round(4)
    display_df['CV_R2_Mean'] = display_df['CV_R2_Mean'].round(4)
    display_df['CV_R2_Std'] = display_df['CV_R2_Std'].round(4)
    display_df['Test_MAE'] = display_df['Test_MAE'].round(2)
    display_df['Test_RMSE'] = display_df['Test_RMSE'].round(2)
    display_df['Overfitting'] = display_df['Overfitting'].round(4)
    
    print(display_df.to_string(index=False))
    
    # Identificar melhor modelo
    best_cost_model = cost_df.iloc[0]
    print(f"\n🏆 MELHOR MODELO PARA CUSTOS: {best_cost_model['Model']}")
    print(f"   R² Score: {best_cost_model['Test_R2']:.4f}")
    print(f"   MAE: R$ {best_cost_model['Test_MAE']:,.2f}")
    print(f"   RMSE: R$ {best_cost_model['Test_RMSE']:,.2f}")

Treinando modelos para predição de custos...

Treinando Linear Regression...
Treinando Ridge Regression...
Treinando Lasso Regression...
Treinando ElasticNet...
Treinando Decision Tree...
Treinando Random Forest...
Treinando Extra Trees...


## 4. Predição de Tempo de Permanência

In [ ]:
if df_clean is not None:
    print("\n=== PREDIÇÃO DE TEMPO DE PERMANÊNCIA ===")
    print()
    
    # Definir features para predição de permanência
    permanence_features = [
        'idade_anos', 'codigo_sexo_encoded', 'urgencia', 'tem_uti', 'gestacao_risco',
        'codigo_especialidade_encoded', 'codigo_complexidade_encoded', 
        'sensivel_atencao_basica', 'mes_competencia', 'idade_grupo'
    ]
    
    # Preparar dados
    X_perm = df_clean[permanence_features].fillna(0)
    y_perm = df_clean['dias_permanencia']
    
    # Dividir dados
    X_perm_train, X_perm_test, y_perm_train, y_perm_test = train_test_split(
        X_perm, y_perm, test_size=0.2, random_state=42
    )
    
    # Normalizar features
    scaler_perm = StandardScaler()
    X_perm_train_scaled = scaler_perm.fit_transform(X_perm_train)
    X_perm_test_scaled = scaler_perm.transform(X_perm_test)
    
    print(f"Features utilizadas: {len(permanence_features)}")
    print(f"Tamanho treino: {len(X_perm_train):,}")
    print(f"Tamanho teste: {len(X_perm_test):,}")
    print(f"Permanência média: {y_perm.mean():.2f} dias")
    print(f"Desvio padrão: {y_perm.std():.2f} dias")
    print()

In [ ]:
if df_clean is not None:
    # Avaliar todos os modelos para predição de permanência
    perm_results = []
    
    print("Treinando modelos para predição de permanência...")
    print()
    
    for name, model in get_regression_models().items():
        print(f"Treinando {name}...")
        
        # Usar dados normalizados para modelos que precisam
        if name in ['SVR', 'K-Neighbors', 'Linear Regression', 'Ridge Regression', 'Lasso Regression', 'ElasticNet']:
            X_train_use = X_perm_train_scaled
            X_test_use = X_perm_test_scaled
        else:
            X_train_use = X_perm_train
            X_test_use = X_perm_test
        
        result = evaluate_model(model, X_train_use, X_test_use, y_perm_train, y_perm_test, name)
        if result:
            perm_results.append(result)
    
    # Criar DataFrame com resultados
    perm_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Predictions'} for r in perm_results])
    perm_df = perm_df.sort_values('Test_R2', ascending=False)
    
    print("\n=== RESULTADOS - PREDIÇÃO DE PERMANÊNCIA ===")
    print()
    
    # Exibir resultados formatados
    display_df = perm_df.copy()
    display_df['Train_R2'] = display_df['Train_R2'].round(4)
    display_df['Test_R2'] = display_df['Test_R2'].round(4)
    display_df['CV_R2_Mean'] = display_df['CV_R2_Mean'].round(4)
    display_df['CV_R2_Std'] = display_df['CV_R2_Std'].round(4)
    display_df['Test_MAE'] = display_df['Test_MAE'].round(2)
    display_df['Test_RMSE'] = display_df['Test_RMSE'].round(2)
    display_df['Overfitting'] = display_df['Overfitting'].round(4)
    
    print(display_df.to_string(index=False))
    
    # Identificar melhor modelo
    best_perm_model = perm_df.iloc[0]
    print(f"\n🏆 MELHOR MODELO PARA PERMANÊNCIA: {best_perm_model['Model']}")
    print(f"   R² Score: {best_perm_model['Test_R2']:.4f}")
    print(f"   MAE: {best_perm_model['Test_MAE']:.2f} dias")
    print(f"   RMSE: {best_perm_model['Test_RMSE']:.2f} dias")

## 5. Visualizações e Análise Comparativa

In [ ]:
if df_clean is not None and len(cost_results) > 0:
    # Gráfico comparativo - Predição de Custos
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Comparação de Modelos - Predição de Custos', fontsize=16, fontweight='bold')
    
    # R² Score
    cost_df_viz = cost_df.head(8)  # Top 8 modelos
    axes[0,0].barh(cost_df_viz['Model'], cost_df_viz['Test_R2'], color='skyblue')
    axes[0,0].set_xlabel('R² Score')
    axes[0,0].set_title('R² Score no Conjunto de Teste')
    axes[0,0].grid(axis='x', alpha=0.3)
    
    # MAE
    axes[0,1].barh(cost_df_viz['Model'], cost_df_viz['Test_MAE'], color='lightcoral')
    axes[0,1].set_xlabel('MAE (R$)')
    axes[0,1].set_title('Erro Médio Absoluto')
    axes[0,1].grid(axis='x', alpha=0.3)
    
    # RMSE
    axes[1,0].barh(cost_df_viz['Model'], cost_df_viz['Test_RMSE'], color='lightgreen')
    axes[1,0].set_xlabel('RMSE (R$)')
    axes[1,0].set_title('Raiz do Erro Quadrático Médio')
    axes[1,0].grid(axis='x', alpha=0.3)
    
    # Cross-validation R²
    axes[1,1].barh(cost_df_viz['Model'], cost_df_viz['CV_R2_Mean'], color='wheat')
    axes[1,1].set_xlabel('R² Score (CV)')
    axes[1,1].set_title('R² Score - Validação Cruzada')
    axes[1,1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Tabela resumo custos
    print("\n📊 RANKING FINAL - PREDIÇÃO DE CUSTOS")
    print("=" * 50)
    for i, row in cost_df.head(5).iterrows():
        print(f"{row.name + 1}º {row['Model']:.<25} R² = {row['Test_R2']:.4f}")

In [ ]:
if df_clean is not None and len(perm_results) > 0:
    # Gráfico comparativo - Predição de Permanência
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Comparação de Modelos - Predição de Permanência', fontsize=16, fontweight='bold')
    
    # R² Score
    perm_df_viz = perm_df.head(8)  # Top 8 modelos
    axes[0,0].barh(perm_df_viz['Model'], perm_df_viz['Test_R2'], color='skyblue')
    axes[0,0].set_xlabel('R² Score')
    axes[0,0].set_title('R² Score no Conjunto de Teste')
    axes[0,0].grid(axis='x', alpha=0.3)
    
    # MAE
    axes[0,1].barh(perm_df_viz['Model'], perm_df_viz['Test_MAE'], color='lightcoral')
    axes[0,1].set_xlabel('MAE (dias)')
    axes[0,1].set_title('Erro Médio Absoluto')
    axes[0,1].grid(axis='x', alpha=0.3)
    
    # RMSE
    axes[1,0].barh(perm_df_viz['Model'], perm_df_viz['Test_RMSE'], color='lightgreen')
    axes[1,0].set_xlabel('RMSE (dias)')
    axes[1,0].set_title('Raiz do Erro Quadrático Médio')
    axes[1,0].grid(axis='x', alpha=0.3)
    
    # Cross-validation R²
    axes[1,1].barh(perm_df_viz['Model'], perm_df_viz['CV_R2_Mean'], color='wheat')
    axes[1,1].set_xlabel('R² Score (CV)')
    axes[1,1].set_title('R² Score - Validação Cruzada')
    axes[1,1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Tabela resumo permanência
    print("\n📊 RANKING FINAL - PREDIÇÃO DE PERMANÊNCIA")
    print("=" * 50)
    for i, row in perm_df.head(5).iterrows():
        print(f"{row.name + 1}º {row['Model']:.<25} R² = {row['Test_R2']:.4f}")

In [ ]:
if df_clean is not None and len(cost_results) > 0 and len(perm_results) > 0:
    # Comparação lado a lado dos R² scores
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    
    # Custos
    cost_top = cost_df.head(8)
    bars1 = ax1.barh(cost_top['Model'], cost_top['Test_R2'], color='lightblue', alpha=0.8)
    ax1.set_xlabel('R² Score')
    ax1.set_title('Predição de Custos', fontsize=14, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for i, bar in enumerate(bars1):
        width = bar.get_width()
        ax1.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', ha='left', va='center', fontsize=9)
    
    # Permanência
    perm_top = perm_df.head(8)
    bars2 = ax2.barh(perm_top['Model'], perm_top['Test_R2'], color='lightcoral', alpha=0.8)
    ax2.set_xlabel('R² Score')
    ax2.set_title('Predição de Permanência', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for i, bar in enumerate(bars2):
        width = bar.get_width()
        ax2.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', ha='left', va='center', fontsize=9)
    
    plt.suptitle('Comparação de Performance - R² Score', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Resumo Final e Conclusões

In [ ]:
if df_clean is not None and len(cost_results) > 0 and len(perm_results) > 0:
    print("\n" + "="*80)
    print("                    RESUMO FINAL DA COMPARAÇÃO")
    print("="*80)
    
    print("\n🎯 MELHORES MODELOS IDENTIFICADOS:")
    print("-" * 40)
    
    # Melhor para custos
    best_cost = cost_df.iloc[0]
    print(f"\n💰 PREDIÇÃO DE CUSTOS:")
    print(f"   Melhor Modelo: {best_cost['Model']}")
    print(f"   R² Score: {best_cost['Test_R2']:.4f}")
    print(f"   MAE: R$ {best_cost['Test_MAE']:,.2f}")
    print(f"   RMSE: R$ {best_cost['Test_RMSE']:,.2f}")
    print(f"   Cross-Val R²: {best_cost['CV_R2_Mean']:.4f} (±{best_cost['CV_R2_Std']:.4f})")
    
    # Melhor para permanência
    best_perm = perm_df.iloc[0]
    print(f"\n⏱️ PREDIÇÃO DE PERMANÊNCIA:")
    print(f"   Melhor Modelo: {best_perm['Model']}")
    print(f"   R² Score: {best_perm['Test_R2']:.4f}")
    print(f"   MAE: {best_perm['Test_MAE']:.2f} dias")
    print(f"   RMSE: {best_perm['Test_RMSE']:.2f} dias")
    print(f"   Cross-Val R²: {best_perm['CV_R2_Mean']:.4f} (±{best_perm['CV_R2_Std']:.4f})")
    
    print("\n📊 ANÁLISE COMPARATIVA:")
    print("-" * 25)
    
    # Contar quantos modelos por tipo tiveram boa performance
    ensemble_models = ['Random Forest', 'Extra Trees', 'Gradient Boosting', 'XGBoost', 'LightGBM']
    linear_models = ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 'ElasticNet']
    tree_models = ['Decision Tree']
    other_models = ['SVR', 'K-Neighbors']
    
    print("\n🏆 TOP 3 MODELOS POR PROBLEMA:")
    print("\nCustos:")
    for i, row in cost_df.head(3).iterrows():
        print(f"   {i+1}. {row['Model']} (R² = {row['Test_R2']:.4f})")
    
    print("\nPermanência:")
    for i, row in perm_df.head(3).iterrows():
        print(f"   {i+1}. {row['Model']} (R² = {row['Test_R2']:.4f})")
    
    print("\n🔍 INSIGHTS:")
    print("-" * 12)
    
    # Análise dos tipos de modelos
    cost_ensemble_avg = cost_df[cost_df['Model'].isin(ensemble_models)]['Test_R2'].mean()
    perm_ensemble_avg = perm_df[perm_df['Model'].isin(ensemble_models)]['Test_R2'].mean()
    
    cost_linear_avg = cost_df[cost_df['Model'].isin(linear_models)]['Test_R2'].mean()
    perm_linear_avg = perm_df[perm_df['Model'].isin(linear_models)]['Test_R2'].mean()
    
    print(f"• Modelos Ensemble têm performance superior:")
    print(f"  - Custos: R² médio = {cost_ensemble_avg:.4f}")
    print(f"  - Permanência: R² médio = {perm_ensemble_avg:.4f}")
    
    print(f"\n• Modelos Lineares são mais simples mas menos precisos:")
    print(f"  - Custos: R² médio = {cost_linear_avg:.4f}")
    print(f"  - Permanência: R² médio = {perm_linear_avg:.4f}")
    
    # Verificar overfitting
    best_cost_overfitting = best_cost['Overfitting']
    best_perm_overfitting = best_perm['Overfitting']
    
    print(f"\n• Análise de Overfitting:")
    print(f"  - {best_cost['Model']} (custos): {best_cost_overfitting:.4f} {'✅ Baixo' if best_cost_overfitting < 0.1 else '⚠️ Moderado' if best_cost_overfitting < 0.2 else '❌ Alto'}")
    print(f"  - {best_perm['Model']} (permanência): {best_perm_overfitting:.4f} {'✅ Baixo' if best_perm_overfitting < 0.1 else '⚠️ Moderado' if best_perm_overfitting < 0.2 else '❌ Alto'}")
    
    print("\n✅ RECOMENDAÇÕES:")
    print("-" * 15)
    print(f"1. Para PRODUÇÃO, use {best_cost['Model']} para custos e {best_perm['Model']} para permanência")
    print(f"2. Considere ensemble voting com os top 3 modelos para maior robustez")
    print(f"3. Monitore performance regularmente e retreine mensalmente")
    print(f"4. Implemente validação cruzada temporal para dados sequenciais")
    
    print("\n" + "="*80)
    
    # Salvar resultados em CSV
    cost_df.to_csv('../data/processed/ml_comparison_costs.csv', index=False)
    perm_df.to_csv('../data/processed/ml_comparison_permanence.csv', index=False)
    print("\n💾 Resultados salvos em: ../data/processed/ml_comparison_*.csv")

## 7. Notas Técnicas

### Metodologia
- **Validação:** 80% treino / 20% teste + validação cruzada 5-fold
- **Pré-processamento:** Normalização para modelos sensíveis à escala
- **Outliers:** Removidos valores extremos (custos > R$ 50.000, permanência > 30 dias)
- **Features:** Variáveis demográficas, clínicas e administrativas

### Limitações
- Dados de 3 meses (Jan-Mar 2025) - considerar sazonalidade
- Região específica (Paraná) - generalização limitada
- Features limitadas ao disponível no DataSUS

### Próximos Passos
1. **Otimização de hiperparâmetros** com Grid/Random Search
2. **Feature engineering** avançada (interações, transformações)
3. **Ensemble methods** combinando melhores modelos
4. **Validação temporal** com dados sequenciais
5. **Explicabilidade** com SHAP/LIME para interpretação